In [ ]:
# hide
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
data = np.load("../assets/freetzan-features.npz")   # features of FreeTZAN's 1,200 clips
genre = data["genre"]                                # labels: 0 to 9, one per genre
train, valid, test = (data["split"] == s for s in ["train", "valid", "test"])


def classifier(X: np.ndarray):
    """A logistic regression classifier, fit to the training clips' features X."""
    model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000))
    return model.fit(X[train], genre[train])


for name in ["waveform", "spectrum_1024", "mfcc"]:
    X = data[name].astype(np.float32)
    model = classifier(X)
    print(f"{name:13s} ({X.shape[1]:4d} values per clip): "
          f"train {model.score(X[train], genre[train]):4.0%}, "
          f"valid {model.score(X[valid], genre[valid]):4.0%}")

In [ ]:
# Tune the STFT frame length on the validation clips. For illustration only, we also
# peek at the test clips, which you should never do in practice!
for n in [256, 512, 1024, 2048]:
    X = data[f"spectrum_{n}"].astype(np.float32)
    model = classifier(X)
    print(f"frame length {n:4d}: valid {model.score(X[valid], genre[valid]):4.0%}, "
          f"test {model.score(X[test], genre[test]):4.0%}")